# VM Resource Planner (Reactive, no forecast)

Notebook này chạy pipeline LP ở chế độ **reactive** cho **CẢ HAI scenarios**:

| Scenario | Objective | Mục tiêu |
|----------|-----------|----------|
| **OVERLOAD** | capacity | Minimize Resource Overload |
| **COST** | cost | Minimize Operational Cost |

## Output Format (Matching PPO)
Output files có format giống PPO với các columns:
- `timestamp`, `allocation`, `vm_cost_per_hour`, `switching_cost`, `total_cost_per_hour`
- `cpu_allocated_cores`, `mem_allocated_gb`, `cpu_vm_only`, `mem_vm_only`
- `cpu_required_cores`, `mem_required_gb`, `cpu_overflow_cores`, `mem_overflow_gb`
- `cpu_utilization_pct`, `mem_utilization_pct`, `sla_violation_flag`

## Output Files
- `lp_schedule_overload.csv` - LP schedule với objective=capacity (format giống PPO)
- `lp_schedule_cost.csv` - LP schedule với objective=cost (format giống PPO)
- `vm_resource_planning_reactive.json` - Combined metrics report

## Các bước chính
1. Load VM catalog (với `cost_per_hour`, `switching_cost`)
2. Load test data từ `cleaned_data.csv` (giống PPO)
3. Chuyển đo đạc → nhu cầu overflow CPU/RAM
4. Giải LP mỗi timestamp cho **cả 2 scenarios**
5. Lưu schedules với format giống PPO

**Sau khi chạy notebook này, chạy `lp_vs_ppo_comparison.ipynb` để so sánh LP vs PPO.**



In [1]:
import json
import importlib
from pathlib import Path

import pandas as pd

# Reload module to ensure latest changes are loaded
import vm_resource_planner
importlib.reload(vm_resource_planner)

from vm_resource_planner import (
    load_vm_catalog,
    load_ground_truth_df,
    convert_forecasts_to_requirements,
    build_reactive_schedule,
    build_reactive_schedule_for_scenario,  # NEW: for both scenarios
    compute_lp_metrics,
    HOST_SPEC,
    VM_TYPES_FILE,
    RESULTS_DIR,
)

RESULTS_DIR.mkdir(exist_ok=True)
print("✓ Libraries & planner helpers loaded (reactive mode)")


✓ Libraries & planner helpers loaded (reactive mode)


In [2]:
vm_catalog = load_vm_catalog(VM_TYPES_FILE)

print("VM catalog (with switching_cost):")
for spec in vm_catalog:
    print(
        f"  • {spec['name']}: {spec['vcpus']} vCPUs, {spec['memory_gb']} GB, "
        f"${spec['cost_per_hour']}/h, switching=${spec['switching_cost']}"
    )

print(f"\nHost spec: {HOST_SPEC}")


VM catalog (with switching_cost):
  • B2s: 2 vCPUs, 4 GB, $0.0416/h, switching=$0.01
  • D2s_v3: 2 vCPUs, 8 GB, $0.096/h, switching=$0.02
  • D8s_v3: 8 vCPUs, 64 GB, $0.384/h, switching=$0.05
  • D32s_v3: 32 vCPUs, 128 GB, $1.536/h, switching=$0.1

Host spec: {'total_cpu_cores': 1, 'total_memory_gb': 4, 'cpu_threshold_pct': 70, 'memory_threshold_pct': 75}


In [3]:
# Load ground truth data từ cleaned_data.csv (giống PPO)
ground_truth_df = load_ground_truth_df()

print(f"✓ Loaded ground truth: {len(ground_truth_df)} rows")
print(f"  Timestamp range: {ground_truth_df['timestamp'].min()} to {ground_truth_df['timestamp'].max()}")
print(f"  Columns: {list(ground_truth_df.columns)}")

print("\nFirst 5 rows:")
display(ground_truth_df.head())


✓ Loaded ground truth: 17150 rows
  Timestamp range: 1970-01-25 01:05:30 to 1970-01-31 00:00:00
  Columns: ['timestamp', 'memory_usage_pct', 'cpu_total_usage', 'system_load']

First 5 rows:


,timestamp,memory_usage_pct,cpu_total_usage,system_load
0,1970-01-25 01:05:30,6.454383,0.0820,0.24
1,1970-01-25 01:06:00,6.454653,0.0570,0.22
2,1970-01-25 01:06:30,6.451089,0.0585,0.13
3,1970-01-25 01:07:00,6.446000,0.0555,0.08
4,1970-01-25 01:07:30,6.458168,0.0635,0.10


In [4]:
# Chuyển ground-truth đo đạc thành nhu cầu overflow
requirements_df = convert_forecasts_to_requirements(ground_truth_df, HOST_SPEC)

print(f"✓ Converted to requirements: {len(requirements_df)} rows")
print(f"  Columns: {list(requirements_df.columns)}")

print("\nSample requirements (first 5 rows):")
display(requirements_df[[
    'timestamp',
    'cpu_total_usage', 'cpu_required_cores', 'cpu_overflow_cores',
    'memory_usage_pct', 'memory_required_gb', 'memory_overflow_gb'
]].head())

# Verify conversion
print("\nVerification (first row):")
row0 = requirements_df.iloc[0]
print(f"  memory_usage_pct: {row0['memory_usage_pct']:.6f} %")
print(f"  mem_required_gb: {row0['memory_required_gb']:.6f} GB")
print(f"  Expected: {HOST_SPEC['total_memory_gb']} GB × {row0['memory_usage_pct']:.6f}% / 100 = {HOST_SPEC['total_memory_gb'] * row0['memory_usage_pct'] / 100.0:.6f} GB")
print(f"  Match: {'✓' if abs(row0['memory_required_gb'] - HOST_SPEC['total_memory_gb'] * row0['memory_usage_pct'] / 100.0) < 1e-6 else '✗'}")


✓ Converted to requirements: 17150 rows
  Columns: ['timestamp', 'memory_usage_pct', 'cpu_total_usage', 'system_load', 'memory_required_gb', 'memory_overflow_gb', 'cpu_required_cores', 'cpu_overflow_cores']

Sample requirements (first 5 rows):


,timestamp,cpu_total_usage,cpu_required_cores,cpu_overflow_cores,memory_usage_pct,memory_required_gb,memory_overflow_gb
0,1970-01-25 01:05:30,0.0820,0.24,0.0,6.454383,0.258175,0.0
1,1970-01-25 01:06:00,0.0570,0.22,0.0,6.454653,0.258186,0.0
2,1970-01-25 01:06:30,0.0585,0.13,0.0,6.451089,0.258044,0.0
3,1970-01-25 01:07:00,0.0555,0.08,0.0,6.446000,0.257840,0.0
4,1970-01-25 01:07:30,0.0635,0.10,0.0,6.458168,0.258327,0.0



Verification (first row):
  memory_usage_pct: 6.454383 %
  mem_required_gb: 0.258175 GB
  Expected: 4 GB × 6.454383% / 100 = 0.258175 GB
  Match: ✓


In [5]:
# Build schedules cho CẢ HAI scenarios
print("Building LP schedules for BOTH scenarios...")
print("=" * 80)

# Scenario 1: OVERLOAD (objective=capacity)
schedule_overload = build_reactive_schedule_for_scenario(requirements_df, vm_catalog, scenario="overload")
print(f"✓ LP Overload: {len(schedule_overload)} steps")
print(f"  Columns: {list(schedule_overload.columns)}")

# Scenario 2: COST (objective=cost)
schedule_cost = build_reactive_schedule_for_scenario(requirements_df, vm_catalog, scenario="cost")
print(f"✓ LP Cost: {len(schedule_cost)} steps")
print(f"  Columns: {list(schedule_cost.columns)}")

# Preview với format giống PPO
print("\n=== LP Overload Schedule Preview (PPO Format) ===")
ppo_format_cols = [
    'timestamp', 'allocation', 'vm_cost_per_hour', 'switching_cost', 'total_cost_per_hour',
    'cpu_allocated_cores', 'mem_allocated_gb', 'cpu_vm_only', 'mem_vm_only',
    'cpu_required_cores', 'mem_required_gb', 'cpu_overflow_cores', 'mem_overflow_gb',
    'cpu_utilization_pct', 'mem_utilization_pct', 'sla_violation_flag'
]
display(schedule_overload[ppo_format_cols].head())

print("\n=== LP Cost Schedule Preview (PPO Format) ===")
display(schedule_cost[ppo_format_cols].head())

# Verify allocation format (should use ':' not '×')
print("\n=== Allocation Format Check ===")
sample_allocs = schedule_overload[schedule_overload['allocation'] != 'Host only']['allocation'].head(3)
for alloc in sample_allocs:
    print(f"  {alloc} {'✓' if ':' in alloc and '×' not in alloc else '✗ (should use : not ×)'}")


Building LP schedules for BOTH scenarios...
✓ LP Overload: 17150 steps
  Columns: ['timestamp', 'allocation', 'vm_cost_per_hour', 'switching_cost', 'total_cost_per_hour', 'cpu_allocated_cores', 'mem_allocated_gb', 'cpu_vm_only', 'mem_vm_only', 'cpu_required_cores', 'mem_required_gb', 'cpu_overflow_cores', 'mem_overflow_gb', 'cpu_utilization_pct', 'mem_utilization_pct', 'sla_violation_flag']
✓ LP Cost: 17150 steps
  Columns: ['timestamp', 'allocation', 'vm_cost_per_hour', 'switching_cost', 'total_cost_per_hour', 'cpu_allocated_cores', 'mem_allocated_gb', 'cpu_vm_only', 'mem_vm_only', 'cpu_required_cores', 'mem_required_gb', 'cpu_overflow_cores', 'mem_overflow_gb', 'cpu_utilization_pct', 'mem_utilization_pct', 'sla_violation_flag']

=== LP Overload Schedule Preview (PPO Format) ===


,timestamp,allocation,vm_cost_per_hour,switching_cost,total_cost_per_hour,cpu_allocated_cores,mem_allocated_gb,cpu_vm_only,mem_vm_only,cpu_required_cores,mem_required_gb,cpu_overflow_cores,mem_overflow_gb,cpu_utilization_pct,mem_utilization_pct,sla_violation_flag
0,1970-01-25 01:05:30,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.24,0.258175,0.0,0.0,0.0,0.0,0
1,1970-01-25 01:06:00,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.22,0.258186,0.0,0.0,0.0,0.0,0
2,1970-01-25 01:06:30,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.13,0.258044,0.0,0.0,0.0,0.0,0
3,1970-01-25 01:07:00,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.08,0.257840,0.0,0.0,0.0,0.0,0
4,1970-01-25 01:07:30,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.10,0.258327,0.0,0.0,0.0,0.0,0



=== LP Cost Schedule Preview (PPO Format) ===


,timestamp,allocation,vm_cost_per_hour,switching_cost,total_cost_per_hour,cpu_allocated_cores,mem_allocated_gb,cpu_vm_only,mem_vm_only,cpu_required_cores,mem_required_gb,cpu_overflow_cores,mem_overflow_gb,cpu_utilization_pct,mem_utilization_pct,sla_violation_flag
0,1970-01-25 01:05:30,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.24,0.258175,0.0,0.0,0.0,0.0,0
1,1970-01-25 01:06:00,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.22,0.258186,0.0,0.0,0.0,0.0,0
2,1970-01-25 01:06:30,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.13,0.258044,0.0,0.0,0.0,0.0,0
3,1970-01-25 01:07:00,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.08,0.257840,0.0,0.0,0.0,0.0,0
4,1970-01-25 01:07:30,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.10,0.258327,0.0,0.0,0.0,0.0,0



=== Allocation Format Check ===
  D32s_v3:1 ✓
  D32s_v3:1 ✓
  D32s_v3:1 ✓


In [6]:
# Compute metrics cho cả hai scenarios
# Note: compute_lp_metrics expects old format, may need update
# For now, compute manually from new format

def compute_metrics_from_ppo_format(df):
    """Compute metrics from PPO-format DataFrame"""
    return {
        "total_vm_cost": float(df["vm_cost_per_hour"].sum()),
        "total_switching_cost": float(df["switching_cost"].sum()),
        "total_cost": float(df["total_cost_per_hour"].sum()),
        "sla_violations": int(df["sla_violation_flag"].sum()),
        "sla_violation_rate": float(df["sla_violation_flag"].mean()),
        "mean_cpu_utilization": float(df["cpu_utilization_pct"].mean()),
        "mean_mem_utilization": float(df["mem_utilization_pct"].mean()),
        "n_steps": len(df),
    }

metrics_overload = compute_metrics_from_ppo_format(schedule_overload)
metrics_cost = compute_metrics_from_ppo_format(schedule_cost)

print("=" * 80)
print("=== LP OVERLOAD Metrics ===")
print("=" * 80)
for k, v in metrics_overload.items():
    print(f"  {k}: {v}")

print("\n" + "=" * 80)
print("=== LP COST Metrics ===")
print("=" * 80)
for k, v in metrics_cost.items():
    print(f"  {k}: {v}")


=== LP OVERLOAD Metrics ===
  total_vm_cost: 605.1840000000001
  total_switching_cost: 15.7
  total_cost: 620.8840000000001
  sla_violations: 0
  sla_violation_rate: 0.0
  mean_cpu_utilization: 0.03478846292974774
  mean_mem_utilization: 0.0
  n_steps: 17150

=== LP COST Metrics ===
  total_vm_cost: 16.3904
  total_switching_cost: 1.57
  total_cost: 17.9604
  sla_violations: 0
  sla_violation_rate: 0.0
  mean_cpu_utilization: 0.5566154068759639
  mean_mem_utilization: 0.0
  n_steps: 17150


In [7]:
# Lưu kết quả cho CẢ HAI scenarios (với format giống PPO)
# Paths
schedule_overload_path = RESULTS_DIR / "lp_schedule_overload.csv"
schedule_cost_path = RESULTS_DIR / "lp_schedule_cost.csv"
report_path = RESULTS_DIR / "vm_resource_planning_reactive.json"

# Save schedules với format PPO (đầy đủ columns)
# Đảm bảo columns theo đúng thứ tự như PPO
ppo_columns = [
    'timestamp', 'allocation', 'vm_cost_per_hour', 'switching_cost', 'total_cost_per_hour',
    'cpu_allocated_cores', 'mem_allocated_gb', 'cpu_vm_only', 'mem_vm_only',
    'cpu_required_cores', 'mem_required_gb', 'cpu_overflow_cores', 'mem_overflow_gb',
    'cpu_utilization_pct', 'mem_utilization_pct', 'sla_violation_flag'
]

schedule_overload[ppo_columns].to_csv(schedule_overload_path, index=False)
schedule_cost[ppo_columns].to_csv(schedule_cost_path, index=False)

print(f"✓ Saved LP Overload: {schedule_overload_path}")
print(f"  Rows: {len(schedule_overload)}, Columns: {len(ppo_columns)}")
print(f"✓ Saved LP Cost: {schedule_cost_path}")
print(f"  Rows: {len(schedule_cost)}, Columns: {len(ppo_columns)}")

# Save combined JSON report
report_payload = {
    'model': 'vm_resource_planner_notebook_reactive',
    'timestamp': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
    'host_spec': HOST_SPEC,
    'scenarios': {
        'overload': {
            'metrics': metrics_overload,
        },
        'cost': {
            'metrics': metrics_cost,
        }
    }
}

with open(report_path, 'w') as f:
    json.dump(report_payload, f, indent=2)

print(f"✓ Saved JSON Report: {report_path}")


✓ Saved LP Overload: E:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure\forecast_result\lp_schedule_overload.csv
  Rows: 17150, Columns: 16
✓ Saved LP Cost: E:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure\forecast_result\lp_schedule_cost.csv
  Rows: 17150, Columns: 16
✓ Saved JSON Report: E:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure\forecast_result\vm_resource_planning_reactive.json


## Visualization

Các biểu đồ sau đây giúp hiểu rõ kết quả chuyển đổi forecast thành VM requirements và allocation schedule.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Set style
plt.style.use('default')
fig_size = (16, 10)

# Convert timestamp to datetime if needed
requirements_df['timestamp'] = pd.to_datetime(requirements_df['timestamp'])
schedule_overload['timestamp'] = pd.to_datetime(schedule_overload['timestamp'])
schedule_cost['timestamp'] = pd.to_datetime(schedule_cost['timestamp'])

# Sample data for visualization (every 100th point for clarity)
sample_idx = requirements_df.index[::100]
req_sample = requirements_df.iloc[sample_idx].copy()
overload_sample = schedule_overload.iloc[sample_idx].copy()
cost_sample = schedule_cost.iloc[sample_idx].copy()

print("✓ Data prepared for visualization")


In [ ]:
# Figure 1: Overflow Values Over Time
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('Resource Overflow Values Over Time', fontsize=16, fontweight='bold')

# CPU Overflow
ax1 = axes[0]
ax1.plot(req_sample['timestamp'], req_sample['cpu_overflow_cores'], 
         linewidth=1.5, alpha=0.7, color='#d62728', label='CPU Overflow')
ax1.axhline(y=0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax1.fill_between(req_sample['timestamp'], 0, req_sample['cpu_overflow_cores'], 
                  alpha=0.3, color='#d62728')
ax1.set_ylabel('CPU Overflow (cores)', fontsize=12)
ax1.set_title('CPU Overflow: Required cores exceeding host threshold (70%)', fontsize=13)
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=11)

# Memory Overflow
ax2 = axes[1]
ax2.plot(req_sample['timestamp'], req_sample['memory_overflow_gb'], 
         linewidth=1.5, alpha=0.7, color='#2ca02c', label='Memory Overflow')
ax2.axhline(y=0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax2.fill_between(req_sample['timestamp'], 0, req_sample['memory_overflow_gb'], 
                  alpha=0.3, color='#2ca02c')
ax2.set_xlabel('Timestamp', fontsize=12)
ax2.set_ylabel('Memory Overflow (GB)', fontsize=12)
ax2.set_title('Memory Overflow: Required GB exceeding host threshold (75%)', fontsize=13)
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=11)

plt.tight_layout()
plt.show()

print("Nhận xét: Overflow values phản ánh các đợt tải cao khi requirements vượt ngưỡng host.")
print("CPU overflow thường biến động mạnh hơn Memory overflow do system load có thể tăng đột ngột.")
print(f"Phần lớn thời gian overflow = 0: {(req_sample['cpu_overflow_cores'] == 0).sum() / len(req_sample) * 100:.1f}% cho CPU, {(req_sample['memory_overflow_gb'] == 0).sum() / len(req_sample) * 100:.1f}% cho Memory")


In [ ]:
# Figure 2: VM Allocation Schedule Comparison
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('VM Allocation Schedule: Overload vs Cost Scenarios', fontsize=16, fontweight='bold')

# Extract VM count from allocation string
def extract_vm_count(allocation_str):
    if allocation_str == 'Host only':
        return 0
    # Count total VMs from format like "B2s:1, D2s_v3:2"
    total = 0
    for part in str(allocation_str).split(', '):
        if ':' in part:
            try:
                count = int(part.split(':')[1])
                total += count
            except:
                pass
    return total

overload_sample['vm_count'] = overload_sample['allocation'].apply(extract_vm_count)
cost_sample['vm_count'] = cost_sample['allocation'].apply(extract_vm_count)

# VM Count Over Time
ax1 = axes[0]
ax1.plot(overload_sample['timestamp'], overload_sample['vm_count'], 
         linewidth=1.5, alpha=0.7, color='#1f77b4', label='Overload Scenario', marker='o', markersize=2)
ax1.plot(cost_sample['timestamp'], cost_sample['vm_count'], 
         linewidth=1.5, alpha=0.7, color='#ff7f0e', label='Cost Scenario', marker='s', markersize=2)
ax1.set_ylabel('Number of VMs', fontsize=12)
ax1.set_title('VM Count Allocation Over Time', fontsize=13)
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=11)

# Cost Comparison
ax2 = axes[1]
ax2.plot(overload_sample['timestamp'], overload_sample['total_cost_per_hour'], 
         linewidth=1.5, alpha=0.7, color='#1f77b4', label='Overload Scenario', marker='o', markersize=2)
ax2.plot(cost_sample['timestamp'], cost_sample['total_cost_per_hour'], 
         linewidth=1.5, alpha=0.7, color='#ff7f0e', label='Cost Scenario', marker='s', markersize=2)
ax2.set_xlabel('Timestamp', fontsize=12)
ax2.set_ylabel('Total Cost ($/hour)', fontsize=12)
ax2.set_title('Total Cost (VM Cost + Switching Cost) Over Time', fontsize=13)
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=11)

plt.tight_layout()
plt.show()

print("Nhận xét: Cost scenario thường allocate ít VM hơn và có chi phí thấp hơn đáng kể.")
print(f"Overload scenario: Max VMs = {overload_sample['vm_count'].max()}, Avg Cost = ${overload_sample['total_cost_per_hour'].mean():.4f}/h")
print(f"Cost scenario: Max VMs = {cost_sample['vm_count'].max()}, Avg Cost = ${cost_sample['total_cost_per_hour'].mean():.4f}/h")


In [ ]:
# Figure 3: Resource Utilization and SLA Violations
fig, axes = plt.subplots(3, 1, figsize=(16, 12))
fig.suptitle('Resource Utilization and SLA Compliance', fontsize=16, fontweight='bold')

# CPU Utilization
ax1 = axes[0]
ax1.plot(overload_sample['timestamp'], overload_sample['cpu_utilization_pct'], 
         linewidth=1.5, alpha=0.7, color='#1f77b4', label='Overload Scenario', marker='o', markersize=2)
ax1.plot(cost_sample['timestamp'], cost_sample['cpu_utilization_pct'], 
         linewidth=1.5, alpha=0.7, color='#ff7f0e', label='Cost Scenario', marker='s', markersize=2)
ax1.axhline(y=100, color='red', linestyle='--', linewidth=1, alpha=0.5, label='100% Utilization')
ax1.set_ylabel('CPU Utilization (%)', fontsize=12)
ax1.set_title('CPU Utilization of Allocated VMs', fontsize=13)
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=11)
ax1.set_ylim([0, 110])

# Memory Utilization
ax2 = axes[1]
ax2.plot(overload_sample['timestamp'], overload_sample['mem_utilization_pct'], 
         linewidth=1.5, alpha=0.7, color='#1f77b4', label='Overload Scenario', marker='o', markersize=2)
ax2.plot(cost_sample['timestamp'], cost_sample['mem_utilization_pct'], 
         linewidth=1.5, alpha=0.7, color='#ff7f0e', label='Cost Scenario', marker='s', markersize=2)
ax2.axhline(y=100, color='red', linestyle='--', linewidth=1, alpha=0.5, label='100% Utilization')
ax2.set_ylabel('Memory Utilization (%)', fontsize=12)
ax2.set_title('Memory Utilization of Allocated VMs', fontsize=13)
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=11)
ax2.set_ylim([0, 110])

# SLA Violations
ax3 = axes[2]
ax3.plot(overload_sample['timestamp'], overload_sample['sla_violation_flag'], 
         linewidth=1.5, alpha=0.7, color='#1f77b4', label='Overload Scenario', marker='o', markersize=3)
ax3.plot(cost_sample['timestamp'], cost_sample['sla_violation_flag'], 
         linewidth=1.5, alpha=0.7, color='#ff7f0e', label='Cost Scenario', marker='s', markersize=3)
ax3.set_xlabel('Timestamp', fontsize=12)
ax3.set_ylabel('SLA Violation Flag', fontsize=12)
ax3.set_title('SLA Violations (1 = Violated, 0 = Compliant)', fontsize=13)
ax3.grid(True, alpha=0.3)
ax3.legend(fontsize=11)
ax3.set_ylim([-0.1, 1.1])

plt.tight_layout()
plt.show()

print("Nhận xét: Utilization thấp cho thấy VM được allocate với buffer lớn.")
print(f"Overload scenario: Mean CPU Util = {overload_sample['cpu_utilization_pct'].mean():.2f}%, Mean Mem Util = {overload_sample['mem_utilization_pct'].mean():.2f}%")
print(f"Cost scenario: Mean CPU Util = {cost_sample['cpu_utilization_pct'].mean():.2f}%, Mean Mem Util = {cost_sample['mem_utilization_pct'].mean():.2f}%")
print(f"SLA Violations: Overload = {overload_sample['sla_violation_flag'].sum()}, Cost = {cost_sample['sla_violation_flag'].sum()}")


In [ ]:
# Figure 4: Cost Breakdown Comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Cost Breakdown: VM Cost vs Switching Cost', fontsize=16, fontweight='bold')

scenarios = ['Overload', 'Cost']
vm_costs = [metrics_overload['total_vm_cost'], metrics_cost['total_vm_cost']]
switch_costs = [metrics_overload['total_switching_cost'], metrics_cost['total_switching_cost']]
total_costs = [metrics_overload['total_cost'], metrics_cost['total_cost']]

x = np.arange(len(scenarios))
width = 0.35

# Stacked bar chart
ax1 = axes[0]
bars1 = ax1.bar(x, vm_costs, width, label='VM Cost', color='#1f77b4', alpha=0.8)
bars2 = ax1.bar(x, switch_costs, width, bottom=vm_costs, label='Switching Cost', color='#ff7f0e', alpha=0.8)
ax1.set_ylabel('Total Cost ($)', fontsize=12)
ax1.set_title('Total Cost Breakdown', fontsize=13)
ax1.set_xticks(x)
ax1.set_xticklabels(scenarios)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3, axis='y')

# Add value labels
for i, (vm, sw, tot) in enumerate(zip(vm_costs, switch_costs, total_costs)):
    ax1.text(i, vm/2, f'${vm:.2f}', ha='center', va='center', fontsize=10, fontweight='bold')
    ax1.text(i, vm + sw/2, f'${sw:.2f}', ha='center', va='center', fontsize=10, fontweight='bold')
    ax1.text(i, tot + tot*0.05, f'Total: ${tot:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Cost per step comparison
ax2 = axes[1]
avg_vm_cost = [vm_costs[i] / metrics_overload['n_steps'] for i in range(2)]
avg_switch_cost = [switch_costs[i] / metrics_overload['n_steps'] for i in range(2)]
avg_total_cost = [total_costs[i] / metrics_overload['n_steps'] for i in range(2)]

bars3 = ax2.bar(x - width/2, avg_vm_cost, width, label='Avg VM Cost/Step', color='#1f77b4', alpha=0.8)
bars4 = ax2.bar(x + width/2, avg_switch_cost, width, label='Avg Switch Cost/Step', color='#ff7f0e', alpha=0.8)
ax2.set_ylabel('Average Cost per Step ($)', fontsize=12)
ax2.set_title('Average Cost per Timestep', fontsize=13)
ax2.set_xticks(x)
ax2.set_xticklabels(scenarios)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("Nhận xét: Cost scenario có tổng chi phí thấp hơn đáng kể do allocate ít VM hơn.")
print(f"Overload scenario: VM Cost = ${metrics_overload['total_vm_cost']:.2f}, Switching = ${metrics_overload['total_switching_cost']:.2f}")
print(f"Cost scenario: VM Cost = ${metrics_cost['total_vm_cost']:.2f}, Switching = ${metrics_cost['total_switching_cost']:.2f}")
print(f"Cost reduction: {((metrics_overload['total_cost'] - metrics_cost['total_cost']) / metrics_overload['total_cost'] * 100):.1f}%")


In [ ]:
# Figure 5: Allocation Pattern Analysis
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('VM Allocation Pattern Analysis', fontsize=16, fontweight='bold')

# Extract VM types from allocation
def get_vm_types(allocation_str):
    if allocation_str == 'Host only':
        return {}
    vm_types = {}
    for part in str(allocation_str).split(', '):
        if ':' in part:
            try:
                vm_name, count = part.split(':')
                vm_types[vm_name] = int(count)
            except:
                pass
    return vm_types

# Count VM type usage
vm_type_counts_overload = {}
vm_type_counts_cost = {}

for alloc in overload_sample['allocation']:
    vm_types = get_vm_types(alloc)
    for vm_name, count in vm_types.items():
        vm_type_counts_overload[vm_name] = vm_type_counts_overload.get(vm_name, 0) + count

for alloc in cost_sample['allocation']:
    vm_types = get_vm_types(alloc)
    for vm_name, count in vm_types.items():
        vm_type_counts_cost[vm_name] = vm_type_counts_cost.get(vm_name, 0) + count

# VM Type Usage
ax1 = axes[0]
vm_names = sorted(set(list(vm_type_counts_overload.keys()) + list(vm_type_counts_cost.keys())))
overload_counts = [vm_type_counts_overload.get(name, 0) for name in vm_names]
cost_counts = [vm_type_counts_cost.get(name, 0) for name in vm_names]

x = np.arange(len(vm_names))
width = 0.35
bars1 = ax1.bar(x - width/2, overload_counts, width, label='Overload Scenario', color='#1f77b4', alpha=0.8)
bars2 = ax1.bar(x + width/2, cost_counts, width, label='Cost Scenario', color='#ff7f0e', alpha=0.8)
ax1.set_ylabel('Total VM Instances Allocated', fontsize=12)
ax1.set_title('VM Type Usage Across All Timesteps', fontsize=13)
ax1.set_xticks(x)
ax1.set_xticklabels(vm_names)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3, axis='y')

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax1.text(bar.get_x() + bar.get_width()/2., height,
                    f'{int(height)}', ha='center', va='bottom', fontsize=9)

# Allocation frequency (how often VMs are allocated)
ax2 = axes[1]
overload_freq = (overload_sample['allocation'] != 'Host only').sum() / len(overload_sample) * 100
cost_freq = (cost_sample['allocation'] != 'Host only').sum() / len(cost_sample) * 100

categories = ['Host Only', 'VM Allocated']
overload_freqs = [100 - overload_freq, overload_freq]
cost_freqs = [100 - cost_freq, cost_freq]

x = np.arange(len(categories))
bars1 = ax2.bar(x - width/2, overload_freqs, width, label='Overload Scenario', color='#1f77b4', alpha=0.8)
bars2 = ax2.bar(x + width/2, cost_freqs, width, label='Cost Scenario', color='#ff7f0e', alpha=0.8)
ax2.set_ylabel('Percentage of Time (%)', fontsize=12)
ax2.set_title('Allocation Frequency: Host Only vs VM Allocated', fontsize=13)
ax2.set_xticks(x)
ax2.set_xticklabels(categories)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3, axis='y')
ax2.set_ylim([0, 105])

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax2.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("Nhận xét: Cost scenario sử dụng VM ít hơn và chủ yếu sử dụng VM nhỏ hơn (B2s, D2s_v3) để minimize cost.")
print(f"Overload scenario: Allocate VM {overload_freq:.1f}% thời gian")
print(f"Cost scenario: Allocate VM {cost_freq:.1f}% thời gian")


In [8]:
# Hoàn tất - Summary
print("="*80)
print("           LP REACTIVE COMPLETED (Both Scenarios)")
print("="*80)

print("\n📁 Output files (PPO format):")
print(f"  • LP Overload: {schedule_overload_path}")
print(f"  • LP Cost:     {schedule_cost_path}")
print(f"  • JSON Report: {report_path}")

print("\n📊 Summary:")
print(f"  {'Metric':<35} {'OVERLOAD':>18} {'COST':>18}")
print("-"*75)
print(f"  {'Total Steps':<35} {metrics_overload['n_steps']:>18} {metrics_cost['n_steps']:>18}")
print(f"  {'VM Cost ($/h sum)':<35} {metrics_overload['total_vm_cost']:>18.4f} {metrics_cost['total_vm_cost']:>18.4f}")
print(f"  {'Switching Cost ($)':<35} {metrics_overload['total_switching_cost']:>18.4f} {metrics_cost['total_switching_cost']:>18.4f}")
print(f"  {'Total Cost ($/h + switch)':<35} {metrics_overload['total_cost']:>18.4f} {metrics_cost['total_cost']:>18.4f}")
print(f"  {'Mean CPU Util %':<35} {metrics_overload['mean_cpu_utilization']:>18.2f} {metrics_cost['mean_cpu_utilization']:>18.2f}")
print(f"  {'Mean Mem Util %':<35} {metrics_overload['mean_mem_utilization']:>18.2f} {metrics_cost['mean_mem_utilization']:>18.2f}")
print(f"  {'SLA Violations':<35} {metrics_overload['sla_violations']:>18} {metrics_cost['sla_violations']:>18}")
print(f"  {'SLA Violation Rate %':<35} {metrics_overload['sla_violation_rate']*100:>18.2f} {metrics_cost['sla_violation_rate']*100:>18.2f}")

# So sánh giữa 2 scenarios
if metrics_overload['total_cost'] > 0 and metrics_cost['total_cost'] > 0:
    diff_pct = (metrics_cost['total_cost'] - metrics_overload['total_cost']) / metrics_overload['total_cost'] * 100
    print(f"\n💡 COST scenario {'đắt hơn' if diff_pct > 0 else 'rẻ hơn'} OVERLOAD: {abs(diff_pct):.1f}%")

print("\n" + "="*80)
print("✓ Output format matches PPO format!")
print("✓ Timestamp from cleaned_data.csv (matching PPO)")
print("✓ Allocation format uses ':' (matching PPO)")
print("✓ All columns present: timestamp, allocation, vm_cost_per_hour, switching_cost,")
print("  total_cost_per_hour, cpu_allocated_cores, mem_allocated_gb, cpu_vm_only,")
print("  mem_vm_only, cpu_required_cores, mem_required_gb, cpu_overflow_cores,")
print("  mem_overflow_gb, cpu_utilization_pct, mem_utilization_pct, sla_violation_flag")
print("="*80)
print("\n✓ Giờ có thể chạy lp_vs_ppo_comparison.ipynb để so sánh LP vs PPO!")


           LP REACTIVE COMPLETED (Both Scenarios)

📁 Output files (PPO format):
  • LP Overload: E:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure\forecast_result\lp_schedule_overload.csv
  • LP Cost:     E:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure\forecast_result\lp_schedule_cost.csv
  • JSON Report: E:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure\forecast_result\vm_resource_planning_reactive.json

📊 Summary:
  Metric                                        OVERLOAD               COST
---------------------------------------------------------------------------
  Total Steps                                      17150              17150
  VM Cost ($/h sum)                             605.1840            16.3904
  Switching Cost ($)                             15.7000             1.5700
  Total Cost ($/h + switch)                     620.8840            17.9604
  Mean CPU Util %            